Setup

In [121]:
!pip3 install statsmodels
!pip3 install linearmodels

In [122]:
import pandas as pd
import statsmodels.api as sm
import numpy as np

Load Data

In [123]:
# Adjust path for file location
PRODUCTS_FILE_PATH = r'/Users/gigisilver/Desktop/products.tsv'
SCANNER1_FILE_PATH = r'/Users/gigisilver/Desktop/3592_2018.tsv'
STORES_FILE_PATH = r'/Users/gigisilver/Desktop/storesScanner_2018.tsv'

# Load Data
products = pd.read_csv (PRODUCTS_FILE_PATH, sep='\t', encoding='latin1')
scanner1 = pd.read_csv (SCANNER1_FILE_PATH, sep='\t', encoding='latin1') 
stores = pd.read_csv (STORES_FILE_PATH, sep='\t', encoding='latin1')

# Sample data if needed
SAMPLE_RATE = 0.1
stores = stores.sample(frac=SAMPLE_RATE, replace=True, random_state=1)

Create Merged Data Set

In [124]:
# Join the products data with the scanner1 data on upc
products_scanner1_df = pd.merge(products, scanner1, how='inner', on = 'upc')

# Create a new data frame which restricts data set to only milk purchases 
# by only including rows where the product group code = the product group description of "MILK".
onlymilk_purchases_df = products_scanner1_df.loc[products_scanner1_df['product_group_code'].isin([2506, 1012])]

# Join the milk purchases data with the stores data on store_code_uc.
final_df = pd.merge(onlymilk_purchases_df, stores, how='inner', on = 'store_code_uc')

Adjust Price for Quantity and Size

In [127]:
# Make a new column PricePerOneProduct which is equal to total price paid / quantity.
final_df["PricePerOneProduct"] = final_df["price"] / final_df["units"]

# Make a new column PricePerOZ which is equal to the PricePerOneProduct / the size amount (number of oz).
final_df["PricePerOZ"] = final_df["PricePerOneProduct"] / final_df["size1_amount"]

Create Regression for Store-Level Fixed Effects

In [129]:
# Only set index if we haven't already
if not 'week_end' in final_df.index.names:
   final_df["week_end"] = pd.to_datetime(final_df.week_end.astype(str))
   final_df = final_df.set_index(["upc", "week_end"])

In [181]:
# Create new collumn 
final_df["week_end_2"] = final_df["week_end"]

In [182]:
# Factorize upc number and week_end number in new columns (reggression can't handle large numbers and will error otherwise)
final_df['upc2']=pd.factorize(final_df['upc'].tolist())[0]
final_df['week_end_2']=pd.factorize(final_df['week_end_2'].tolist())[0]

In [183]:
from linearmodels.panel import PanelOLS

exog_vars = ["store_code_uc", "upc2"]
exog = sm.add_constant(final_df[exog_vars])
mod = PanelOLS(final_df.PricePerOZ, exog, entity_effects=True)
fe_res = mod.fit()
print(fe_res)


ValueError: Series can only be used with a 2-level MultiIndex

Find Chains
- Valid chains are those in which at least 80% of stores with that retailer_code have the same parent_code according to Uniform Pricing in U.S. Retail Chains Paper

In [131]:
# Group stores dataset by retailer_code and then parent_code
# Count number of uniqe retailer_code parent_code combonations and create new column "store_count" with this count
combo_count_df = stores.groupby(['retailer_code','parent_code'])['store_code_uc'].count().reset_index(name="store_count")
#pd.set_option('display.max_rows', 10)
#combo_count

In [132]:
# Create new column "parent_total" 
# Group store_count by parent_code and find the total number of the same parent_code for each retaoler code
combo_count_df['parent_total'] = combo_count_df.groupby(['parent_code'])['store_count'].transform('sum').reset_index(name="parent_total")['parent_total']

In [133]:
# Create new column = to the store_count/parent_total
combo_count_df["same_parent_percent"] = combo_count_df["store_count"] / combo_count_df["parent_total"]

# Create a new column "is_chain" which returns true if the "same_parent_percent" is greater than 80%
# This will allow us to know if a retialer is apart of a chain by definition of the Uniform Pricing in U.S. Retail Chains Paper  
combo_count_df["is_chain"] = combo_count_df['same_parent_percent'] > 0.8
#combo_count.loc[combo_count['same_parent_percent'] > 0.8]

In [172]:
# Set is_chain to a 0 or 1 instead of true or false 
i=0
for x in combo_count_df["is_chain"]:
    combo_count_df["is_chain"][i] = 0 if not x else 1
    i+=1

/var/folders/jn/w23_jq6x5bl_00592v8hyv0r0000gn/T/ipykernel_1808/938908258.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combo_count_df["is_chain"][i] = 0 if not x else 1


In [170]:
# Set is_chain to a 0 or 1 instead of true or false 
combo_count_df['is_chain'] = np.where(combo_count_df['is_chain'], 1,  0)

Create Regression for Chain-Level Fixed Effects

In [173]:
# Reset index on final_df so that purchase_date is included in chain_df
final_df = final_df.reset_index()

# Merge combo_count_df with final_df before running regression
chain_df = pd.merge(combo_count_df, final_df, how='inner', on = 'retailer_code')

In [174]:
# Create new collumn 
chain_df["week_end_2"] = chain_df["week_end"]

In [175]:
# Set index for Panel Regression
if not 'week_end' in chain_df.index.names:
    chain_df["week_end"] = pd.to_datetime(chain_df.week_end.astype(str))
    chain_df = chain_df.set_index(["store_code_uc", "week_end"])

In [177]:
# Factorize upc number and week_end number in new columns (reggression can't handle large numbers and will error otherwise)
chain_df['upc2']=pd.factorize(chain_df['upc'].tolist())[0]
chain_df['week_end_2']=pd.factorize(chain_df['week_end_2'].tolist())[0]

In [178]:
from linearmodels.panel import PanelOLS

exog_vars = ["is_chain", "upc2", "week_end_2"]
exog = sm.add_constant(chain_df[exog_vars])
mod = PanelOLS(chain_df.PricePerOZ, exog, entity_effects=True, time_effects=True)
fe_res = mod.fit()
print(fe_res)
#check_rank=False, drop_absorbed=True

AbsorbingEffectError: 
The model cannot be estimated. The included effects have fully absorbed
one or more of the variables. This occurs when one or more of the dependent
variable is perfectly explained using the effects included in the model.

The following variables or variable combinations have been fully absorbed
or have become perfectly collinear after effects are removed:

          const, is_chain, week_end_2

Set drop_absorbed=True to automatically drop absorbed variables.


In [ ]:
#y = chain_df[['upc','is_chain']]
#y = y.reset_index()
#y = y.drop(columns=['store_code_uc', 'week_end'])
#y.dtypes

upc          int64
is_chain    object
dtype: object

In [ ]:
#y['is_chain_3'] = np.where(y['is_chain'], 1,  0)
#y.corr()


,upc,is_chain_3
upc,1.000000,-0.075445
is_chain_3,-0.075445,1.000000
